# Task 2 - End-to-End ML Pipeline for Customer Churn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib


In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes':1,'No':0})


In [ ]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns
preprocessor = ColumnTransformer([
('num', Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]), num_cols),
('cat', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])


In [ ]:
pipeline = Pipeline([
('preprocessor', preprocessor),
('classifier', LogisticRegression(max_iter=1000))
])
params = {'classifier__C':[0.1,1,10]}
grid = GridSearchCV(pipeline, params, cv=3, scoring='accuracy')
grid.fit(X,y)
joblib.dump(grid.best_estimator_, 'churn_pipeline.pkl')
print(grid.best_params_)


In [ ]:
rf_pipeline = Pipeline([
('preprocessor', preprocessor),
('classifier', RandomForestClassifier())
])
rf_params = {'classifier__n_estimators':[100,200]}
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=3)
rf_grid.fit(X,y)
print(rf_grid.best_params_)
